In [13]:
# controlling the output
'''
Use message prefilling and stop sequences only to get three different commands in a single response.
There shouldn't be any comments or explanation.
Message prefilling isn't limited to just characters like\'\'\'
'''

"\nUse message prefilling and stop sequences only to get three different commands in a single response.\nThere shouldn't be any comments or explanation.\nMessage prefilling isn't limited to just characters like'''\n"

In [14]:
from anthropic import Anthropic
from dotenv import load_dotenv
load_dotenv()

model = "claude-haiku-4-5-20251001"
client = Anthropic()

In [15]:
# 3 helper functions
def add_user_message(messages, text):
    user_message = {
        "role": "user",
        "content": text
    }
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {
        "role": "assistant",
        "content": text
    }
    messages.append(assistant_message)

def chat(messages, system = None, stop_sequences = []): # new parameters for the code
    params = {
        "model" : model,
        "max_tokens" : 1000,
        "messages": messages,
        "stop_sequences": stop_sequences
    }

    if system:
        params['system'] = system
    message = client.messages.create(**params)

    return message.content[0].text

In [16]:
messages = []
prompt = """
Generate three different sample AWS CLI commands. Each should be very short.
"""

add_user_message(messages, prompt)
add_assistant_message(messages, "Here are all three commands in a single blocks without any comments:\n ```bash")
answer = chat(messages, stop_sequences = ["```"])
print(answer)
add_assistant_message(messages, answer)


aws s3 ls
aws ec2 describe-instances
aws lambda list-functions



### Prompt Evaulation Code

In [17]:
import json

def generate_dataset():
    prompt = """
    Generate an evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts that generate Python, JSON or Regex specifically for AWS-related tasks. Generate an array of JSON code each representing the task that requires Python, JSON or a Regex to complete.

    Example output:
    ```json
    [
        {
            "task": "Description of task",
        },
        ...additional
    ]
    ```
    * Focus on tasks that can be solved by writing a single Python function, a single JSON object or a regular expression.
    * Focus on tasks that do not require writing much code.

    Please generate 3 objects
    """
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, '```json')
    text = chat(messages, stop_sequences=['```'])
    return json.loads(text)

In [18]:
dataset = generate_dataset() # create the dataset

# write the json in a json file usign dump (Python to JSON)
with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent = 2)

### Running the eval

In [19]:
def run_prompt(test_case):
    # Merges the prompt and test case input, then returns the result.
    prompt = f"""
    Please solve the following task:
    {test_case["task"]}
    """

    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

In [20]:
def run_test_case(test_case):
    # Calls run_prompt, then grades the result.
    output = run_prompt(test_case)

    # TODO: Grading
    score = 10

    # return a dictionary
    return {
        "output": output,
        "test_case": test_case,
        "score": score
    }

In [21]:
def run_eval(dataset):
    # Loads the dataset and calls run_test_case with each case.
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    
    return results

In [22]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

In [24]:
print(json.dumps(results,indent=2))

[
  {
    "output": "# S3 Read-Only IAM Policy\n\nHere's a JSON policy document that allows an IAM user to read objects from the S3 bucket named 'my-data-bucket':\n\n```json\n{\n  \"Version\": \"2012-10-17\",\n  \"Statement\": [\n    {\n      \"Sid\": \"ListBucketContents\",\n      \"Effect\": \"Allow\",\n      \"Action\": [\n        \"s3:ListBucket\"\n      ],\n      \"Resource\": \"arn:aws:s3:::my-data-bucket\"\n    },\n    {\n      \"Sid\": \"ReadObjectsFromBucket\",\n      \"Effect\": \"Allow\",\n      \"Action\": [\n        \"s3:GetObject\",\n        \"s3:GetObjectVersion\"\n      ],\n      \"Resource\": \"arn:aws:s3:::my-data-bucket/*\"\n    }\n  ]\n}\n```\n\n## Explanation\n\n| Component | Purpose |\n|-----------|---------|\n| **ListBucket** | Allows the user to list contents of the bucket |\n| **GetObject** | Allows the user to read/download objects from the bucket |\n| **GetObjectVersion** | Allows access to specific object versions (if versioning is enabled) |\n| **Resource A